# OpenBind-HIPPO

- **Target: D68EV3C**
- **Cycle: 01**

## Prep

- [x] Downloaded D68EV3C with fragalysis_download.ipynb
- [x] Bulkdock setup: `python -m bulkdock setup D68EV3C`
- [ ] Copy aligned files to here: `cp -rv $BULK/TARGETS/D68EV3C/aligned_files .`

## Imports

In [21]:
%load_ext autoreload
%autoreload 2
import hippo
import mrich
from mrich import print
from pathlib import Path
from os import environ
import shutil

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Config

In [19]:
target_name = "XX01ZVNS2B"
target_dir = Path(environ["BULK"]) / "TARGETS" / target_name
cycle_name = "cycle_01"
cycle_dir = Path(cycle_name)
aligned_dir = target_dir / "aligned_files"

## Animal

In [14]:
animal = hippo.HIPPO(target_name, target_dir / f"{target_name}.sqlite")

 Creating HIPPO animal

name = XX01ZVNS2B

db_path = /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/XX01ZVNS2B/XX01ZVNS2B.sqlite

DEBUG: hippo.Database.__init__()

DEBUG: Database.path = /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/XX01ZVNS2B/XX01ZVNS2B.sqlite

DEBUG: hippo.Database.connect()

DEBUG: sqlite3.version='2.6.0'

 Success  Database connected @ /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/XX01ZVNS2B/XX01ZVNS2B.sqlite!

 Success  Initialised animal HIPPO("XX01ZVNS2B")!

## Merging

Run merging algorithms on all poses tagged 'hits'

In [26]:
merge_input_poses = animal.poses(tag="hits")
merge_input_poses

poses tagged "hits": {P × 337}

### Create inputs

- CSV input for Knitwork
- SDF of hits to merge for Knitwork and Fragmenstein
- Protonated PDBs for Knitwork
- Reference PDB for Fragmenstein

In [25]:
# output directories
knitwork_out_dir = cycle_dir / "knitwork"
knitwork_out_dir.mkdir(parents=True, exist_ok=True)
fragmenstein_out_dir = cycle_dir / "fragmenstein"
fragmenstein_out_dir.mkdir(parents=True, exist_ok=True)

In [31]:
# knitwork CSV
knitwork_out_dir = cycle_dir / "knitwork"
knitwork_out_dir.mkdir(parents=True, exist_ok=True)
poses.to_knitwork(knitwork_out_dir / f"{cycle_name}_input.csv", path_root=knitwork_out_dir, aligned_files_dir="aligned_files")

out_path = /opt/xchem-fragalysis-2/maxwin/openbind-hippo/xx01zvns2b/cycle_01/knitwork/cycle_01_input.csv

path_root = /opt/xchem-fragalysis-2/maxwin/openbind-hippo/xx01zvns2b/cycle_01/knitwork

aligned_files_dir = aligned_files

 DISK  Writing /opt/xchem-fragalysis-2/maxwin/openbind-hippo/xx01zvns2b/cycle_01/knitwork/cycle_01_input.csv...

In [27]:
# reference apo PDB for Fragmenstein
ref_pose = merge_input_poses[0]
mrich.var("ref_pose", ref_pose)
shutil.copy(ref_pose.apo_path, fragmenstein_out_dir)

ref_pose = C1->P1: "Z0490a"

'cycle_01/fragmenstein/Z0490a_apo-desolv.pdb'

In [30]:
# SDF of hits
out_dir = cycle_dir
out_dir.mkdir(parents=True, exist_ok=True)
merge_input_poses.write_sdf(cycle_dir / f"{cycle_name}_hits.sdf")

Output()

 DISK  Reading /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/XX01ZVNS2B/aligned_files/Z0777a/Z0777a_hippo.pdb...

 Warning  Multiple molecules in SDF P40!

 DISK  Writing cycle_01/cycle_01_hits.sdf...

### Run Fragmenstein

```
cd cycle_01/fragmenstein
sbatch --job-name "xx01zvns2b_fragmenstein" --mem 16000 $HOME2/slurm/run_bash_with_conda.sh run_fragmenstein.sh
```